# PretextGuard Phase 13: train the two N1 models

This notebook fine-tunes DistilBERT twice for the N1 ablation: **model A** reads the raw body of an email (links, addresses and file names as they were) and **model B** reads the redacted body
(`[URL]`, `[EMAIL]`, `[DOMAIN]`, `[FILE]`). Both learn one thing: attack (phishing or fraud) or not. Everything else is identical, so any difference between them on the test emails is the effect of the redaction.
All the training code lives in the project repository (`src/eval/n1_train.py`); the notebook only prepares the Colab machine, starts it and saves the results.

**Before you run it**
1. Runtime > Change runtime type > **T4 GPU**.
2. Your Google Drive has a folder `pretextguard` that holds `n1_data.parquet` (made on the Mac with `python -m src.eval.n1_data`; it holds a training sample and a validation sample, train and validation emails only).
3. The Phase 13 code is pushed to GitHub (the notebook clones it).

Then choose Runtime > Run all. Expect about 40 minutes of GPU time (an estimate: two models, two seeds each, two epochs each, on about 9,000 emails; the time per epoch is printed). Cell 4 is the long one.
The test split never comes near this notebook.

## 1. Check that a GPU is attached

In [ ]:
import torch

print("PyTorch", torch.__version__, "| GPU available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Use Runtime > Change runtime type > T4 GPU, then run again from the top."
print(torch.cuda.get_device_name(0))

## 2. Get the code and pin `transformers`

The repository is public, so it is cloned without a login. `requirements.txt` in the clone says which `transformers` version to install, so Colab and the Mac read the model files with the same library.
`torch` is **not** reinstalled: Colab already has a build that matches its GPU driver. The versions are printed and saved in `n1_run_info.json`, and the Mac checks that its predictions equal Colab's.

In [ ]:
import os, re, subprocess

REPO = "https://github.com/Nagasai-Datta/PretextGuard.git"
BRANCH = "main"
if not os.path.exists("/content/PretextGuard"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/content/PretextGuard"], check=True)
%cd /content/PretextGuard

pins = dict(re.findall(r"^(torch|transformers)==(\S+)", open("requirements.txt").read(), re.M))
print("requirements.txt pins:", pins)
subprocess.run(["pip", "install", "-q", "transformers==" + pins["transformers"]], check=True)
print(subprocess.run(["python", "-c", "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__)"], capture_output=True, text=True).stdout)
print("code version:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
assert os.path.exists("src/eval/n1_train.py"), "src/eval/n1_train.py is not in the clone: push the Phase 13 code to GitHub first."

## 3. Mount Google Drive and check the data

Colab asks for permission to read your Drive. The check prints how many emails of each kind the file holds and stops if a test row is in it: the test split must never reach the training code.

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount("/content/drive")
DRIVE_FOLDER = "/content/drive/MyDrive/pretextguard"
DATA = DRIVE_FOLDER + "/n1_data.parquet"
OUT = "/content/phase13_outputs"

table = pd.read_parquet(DATA)
print(table.groupby(["split", "category"]).size().unstack(fill_value=0))
print("attacks:", table.groupby("split")["is_attack"].sum().to_dict(), "| with a link:", table.groupby("split")["has_url"].sum().to_dict())
assert set(table["split"]) == {"train", "validation"}, "the file must hold train and validation rows only"

## 4. Train

`n1_train.py` trains model A (raw view) and model B (redacted view), each with seeds 42 and 43 for two epochs, and keeps for each model the seed and epoch with the best attack-class F1 on the validation sample.
Every epoch prints the training loss, the validation loss and the validation attack-class F1. Hugging Face may print a warning that the classifier layer is newly initialised: that is expected.

In [ ]:
!python -u -m src.eval.n1_train --data "{DATA}" --out "{OUT}"

## 5. Look at the results

The table says which seed and epoch each model kept. The validation F1 here is on the validation sample (which chose the epoch and the seed), so it is a little optimistic; the test split gives the honest figures, on the Mac.

In [ ]:
import json

info = json.load(open(OUT + "/n1_run_info.json"))
print(json.dumps(info["chosen"], indent=2))
print("minutes:", info["minutes"], "| device:", info["device"], "| fp16:", info["fp16"])
log = pd.read_csv(OUT + "/n1_training_log.csv")
print(log.to_string(index=False))

## 6. Save everything to Drive

The zip holds the two model folders and three small files. Download `phase13_outputs.zip` from the `pretextguard` folder in Google Drive (browser), then follow `notebooks/README.md`, section "Phase 13: back on the Mac".

In [ ]:
import shutil

zip_path = shutil.make_archive(DRIVE_FOLDER + "/phase13_outputs", "zip", OUT)
print(zip_path, round(os.path.getsize(zip_path) / 1e6), "MB")